### Criando banco de dados silver:

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

### Regras de Negócio e Transformações Exigidas:

#### 1. silver.dim_consumidores (origem: tb_customers):

In [0]:
from pyspark.sql.functions import col, upper, row_number
from pyspark.sql.window import Window

#deduplicação sênior: mantendo o registro mais recente:
w = Window.partitionBy("customer_id").orderBy(col("timestamp_ingestion").desc())

#aqui vou começar a mexer nas colunas

df = spark.table("bronze.tb_customers")

#aplica a deduplicação:
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

#modificando os nomes das colunas 
df = df.select(
    col("customer_id").alias("id_consumidor"),
    col("customer_zip_code_prefix").alias("prefixo_cep"),
    col("customer_unique_id").alias("id_consumidor_unico"),
    upper(col("customer_city")).alias("cidade"),
    upper(col("customer_state")).alias("estado")
)

#salvando no deltalake
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.dim_consumidores")

print("silver.dim_consumidores criado!")


silver.dim_consumidores criado!


#### 2. silver.fat_pedidos (origem: tb_orders):

In [0]:
from pyspark.sql.functions import col, when, datediff, to_timestamp


df = spark.table("bronze.tb_orders")

#convertendo a coluna de status pra pt
df = df.withColumn("status", when(col("order_status") == "delivered", "entregue")
    .when(col("order_status") == "canceled", "cancelado")
    .when(col("order_status") == "shipped", "enviado")
    .when(col("order_status") == "processing", "processando")
    .when(col("order_status") == "invoiced", "faturado")
    .when(col("order_status") == "unavailable", "indisponível")
    .when(col("order_status") == "created", "criado")
    .when(col("order_status") == "approved", "aprovado")
    .otherwise(col("order_status")))

#criar novas colunas derivadas:

#tempo_entrega_dias: diferença em dias entre a data de entrega real e a data de compra
df = df.withColumn("tempo_entrega_dias",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))

#tempo_entrega_estimado_dias: diferença em dias entre a data estimada e a data de compra.
df = df.withColumn("tempo_entrega_estimado_dias",
    datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))

#diferenca_entrega_dias: diferença entre o tempo real e o tempo estimado
df = df.withColumn("diferenca_entrega_dias",
    col("tempo_entrega_dias") - col("tempo_entrega_estimado_dias"))

#entrega_no_prazo: indicador textual baseado na coluna de status("Sim", "Não", ou "Não Entregue").
df = df.withColumn("entrega_no_prazo",
    when(col("status") != "entregue", "Não Entregue")
    .when(col("diferenca_entrega_dias") <= 0, "Sim")
    .otherwise("Não"))

#traduzindo o nome das colunas
df = df.select(
    col("order_id").alias("id_pedido"),
    col("customer_id").alias("id_consumidor"),
    col("status"),
    col("order_purchase_timestamp").alias("data_pedido"),
    col("order_approved_at").alias("data_aprovacao"),
    col("order_delivered_customer_date").alias("data_entrega"),
    col("order_estimated_delivery_date").alias("data_entrega_estimada"),"tempo_entrega_dias","tempo_entrega_estimado_dias",
    "diferenca_entrega_dias","entrega_no_prazo"
)



#salvando na delta
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.fat_pedidos")
print("silver.fat_pedidos criado!")



silver.fat_pedidos criado!


#### 3. silver.fat_itens_pedidos (origem: tb_order_items):

In [0]:
df = spark.table("bronze.tb_order_items")

#tradução:
df = df.select(
    col("order_id").alias("id_pedido"),
    col("order_item_id").alias("id_item"),
    col("product_id").alias("id_produto"),
    col("seller_id").alias("id_vendedor"),
    col("price").cast("double").alias("preco_BRL"),          # cast aqui
    col("freight_value").cast("double").alias("preco_frete") # e aqui
)

df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.fat_itens_pedidos")
print("silver.fat_itens_pedidos criado!")


silver.fat_itens_pedidos criado!


#### 4. silver.fat_pagamentos_pedidos (origem: tb_order_payments):


In [0]:
df = spark.table("bronze.tb_order_payments")
#traduzindo o status
df = df.withColumn("tipo_pagamento",
    when(col("payment_type") == "credit_card", "Cartão de Crédito")
    .when(col("payment_type") == "boleto", "Boleto")
    .when(col("payment_type") == "voucher", "Voucher")
    .when(col("payment_type") == "debit_card", "Cartão de Débito")
    .otherwise("Não Definido"))


#traduzindo o nome de cada coluna
df = df.select(
    col("order_id").alias("id_pedido"),
    col("payment_sequential").alias("sequencia_pagamento"),
    col("tipo_pagamento"),
    col("payment_installments").alias("parcelas"),
    col("payment_value").alias("valor_pagamento")
)


df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.fat_pagamentos_pedidos")
print("silver.fat_pagamentos_pedidos criado!")


silver.fat_pagamentos_pedidos criado!


#### 5. silver.fat_avaliacoes_pedidos (origem: tb_order_reviews):


In [0]:
from pyspark.sql.functions import try_to_timestamp, lit, current_date


df = spark.table("bronze.tb_order_reviews")


# #convertendo datas:
df = df.withColumn("data_criacao_avaliacao", try_to_timestamp(col("review_creation_date")))

df = df.withColumn("data_resposta_avaliacao", try_to_timestamp(col("review_answer_timestamp")))


#filtros:
df = df.filter(col("order_id").isNotNull())
df = df.filter(col("data_criacao_avaliacao").isNull() | (col("data_criacao_avaliacao") <= current_date()))


# preechendo os nulos
df = df.fillna({"review_comment_title": "Sem título", "review_comment_message": "Sem comentário"})

#tradução:
df = df.select(
    col("review_id").alias("id_avaliacao"),
    col("order_id").alias("id_pedido"),
    col("review_score").alias("nota_avaliacao"),
    col("review_comment_title").alias("titulo_avaliacao"),
    col("review_comment_message").alias("comentario_avaliacao"),
    col("data_criacao_avaliacao"),
    col("data_resposta_avaliacao")
)

df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.fat_avaliacoes_pedidos")
print("silver.fat_avaliacoes_pedidos criado!")



silver.fat_avaliacoes_pedidos criado!


#### 6. silver.dim_produtos (origem: tb_products):

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, upper, row_number

#deduplicação:
w = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())
df = spark.table("bronze.tb_products")
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

#tradução do nome das colunas:
df = df.select(
    col("product_id").alias("id_produto"),
    col("product_category_name").alias("categoria_produto"),
    col("product_name_lenght").alias("tamanho_nome_produto"),
    col("product_description_lenght").alias("tamanho_descricao_produto"),
    col("product_photos_qty").alias("quantidade_fotos"),
    col("product_weight_g").alias("peso_produto_gramas"),
    col("product_length_cm").alias("comprimento_centimetros"),
    col("product_height_cm").alias("altura_centimetros"),
    col("product_width_cm").alias("largura_centimetros")
)
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.dim_produtos")

print("silver.dim_produtos criado!")


silver.dim_produtos criado!


#### 7. silver.dim_vendedores (origem: tb_sellers):

In [0]:
#deduplicação:
w = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())
df = spark.table("bronze.tb_sellers")
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

#tradução das colunas:
df = df.select(
    col("seller_id").alias("id_vendedor"),col("seller_zip_code_prefix").alias("prefixo_cep"),
    upper(col("seller_city")).alias("cidade"),
    upper(col("seller_state")).alias("estado")
)
#salvando:
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.dim_vendedores")
print("silver.dim_vendedores criado!")



silver.dim_vendedores criado!


#### 8. silver.dim_categoria_produtos_traducao:

In [0]:
df = spark.table("bronze.tb_product_category_name_translation")

#mudando nome das coolunas:
df = df.select(
    col("product_category_name").alias("nome_produto_pt"),
    col("product_category_name_english").alias("nome_produto_en")
)
#salvando
df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.dim_categoria_produtos_traducao")
print("silver.dim_categoria_produtos_traducao criado!")


silver.dim_categoria_produtos_traducao criado!


#### 9. silver.dim_cotacao_dolar:

In [0]:
from pyspark.sql.functions import (to_date, col, last, to_timestamp,
    sequence, explode, lit, date_format)
from pyspark.sql.window import Window
from pyspark.sql.types import DateType

#agrupando todas as cotações do mesmo dia:
df_raw = spark.table("bronze.tb_cotacao_dolar")
df_raw = df_raw.withColumn("data", to_date(col("dataHoraCotacao")))
df_raw = df_raw.groupBy("data").agg({"cotacaoCompra": "last"})
df_raw = df_raw.withColumnRenamed("last(cotacaoCompra)", "cotacao_compra")


#criar um calendário com todas as possiveis datas no intervalo que baixei os dados (ingestao da api)
datas = spark.sql("SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as data")


#junta o calendario criado com as cotações
#leftjoin:
df_cal = datas.join(df_raw, on="data", how="left")
w_fill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)
df_cal = df_cal.withColumn("cotacao_compra", last(col("cotacao_compra"), ignorenulls=True).over(w_fill))


df_cal.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.dim_cotacao_dolar")
print("silver.dim_cotacao_dolar criado!")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver.dim_cotacao_dolar criado!


#### Tabela Final Silver (silver.fat_pedido_total):

In [0]:
from pyspark.sql.functions import round as spark_round, to_date, sum as spark_sum

#lendo as tabelas antes de juntar:
pedidos = spark.table("silver.fat_pedidos").select("id_pedido", "id_consumidor", "status", "data_pedido")

pagamentos = spark.table("silver.fat_pagamentos_pedidos") \
    .groupBy("id_pedido").agg(spark_sum("valor_pagamento").alias("valor_total_pago_brl"))

dolar = spark.table("silver.dim_cotacao_dolar").select(col("data"), col("cotacao_compra"))


#leftjoin: pedidos + pagamento
df = pedidos.join(pagamentos, on="id_pedido", how="left")
df = df.withColumn("data_join", to_date(col("data_pedido")))

#leftjoin dnv: junta data do pedido cm a cotação
df = df.join(dolar, df.data_join == dolar.data, how="left")

#convertendo moeda e ajustando formatação
df = df.withColumn("valor_total_pago_usd",
    spark_round(col("valor_total_pago_brl") / col("cotacao_compra"), 2))
df = df.withColumn("valor_total_pago_brl", spark_round(col("valor_total_pago_brl"), 2))


df = df.select("id_pedido", "id_consumidor", "status",
    "valor_total_pago_brl", "valor_total_pago_usd", "data_pedido")


df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver.fat_pedido_total")
print("silver.fat_pedido_total criado!")



silver.fat_pedido_total criado!


#### Otimização Física (Delta Optimize)

In [0]:
# Otimização física para performance analítica
spark.sql("OPTIMIZE silver.fat_pedidos ZORDER BY (id_pedido, data_pedido)")
spark.sql("OPTIMIZE silver.fat_itens_pedidos ZORDER BY (id_pedido)")
spark.sql("OPTIMIZE silver.fat_pagamentos_pedidos ZORDER BY (id_pedido)")
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")
print("OPTIMIZE concluido")


OPTIMIZE concluido
